importing Required modules

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn import preprocessing as per
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt 
from sklearn_pandas import DataFrameMapper
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.cross_decomposition import PLSRegression
from lifelines.utils import concordance_index
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,median_absolute_error


import deepsurvk
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNetCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error

G:\ana\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
G:\ana\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.8.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported con

In [2]:
#READING CSV FILE
df1 = pd.read_csv("data_bcr_clinical_data_patient.csv",na_values='?')
#EXCEPT CLINICAL DATA OTHERS HAVE PATIENT IDs WITH -01, SO ADD -01 AT THE END
df1.at[4,"Patient Identifier"]
def ankfunc(s):
    return s+"-01"
for i in range(4,532):
    df1.at[i,"Patient Identifier"]=ankfunc(df1.at[i,"Patient Identifier"])

#DROP ROWS AND COLUMNS
df1.drop([0,1,2,3] , inplace=True)
df1.set_index("Patient Identifier", inplace=True)

    
df1.replace('unknown',np.nan , inplace=True)
df1.replace('[Not Available]',np.nan , inplace=True)

df1.fillna(df1.mean(), inplace=True)
df1

C:\Users\PRODEE~1\AppData\Local\Temp/ipykernel_31092/2649448865.py:18: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df1.fillna(df1.mean(), inplace=True)


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NaN,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NaN,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [3]:
df1.fillna(method='ffill', inplace=True)
df1.fillna(method='bfill', inplace=True)
df1

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,1:Recurred/Progressed,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NOT HISPANIC OR LATINO,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NOT HISPANIC OR LATINO,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [4]:
df1['Lymph node neck dissection indicator'].replace(['[Not Available]','NO','YES'],['00','1','2'],inplace=True)

df1['Overall Survival Status'].replace(['0:LIVING','1:DECEASED'],['0','1'],inplace=True)
df1['Patient Primary Tumor Site'].replace(['[Not Available]','Buccal Mucosa','Larynx','Oral Cavity','Floor of mouth','Tonsil','Hypopharynx','Alveolar Ridge','Hard Palate','Oropharynx','Lip','Base of tongue','Oral Tongue'],['00','1','2','3','4','5','6','7','8','9','10','11','12'],inplace=True)
df1['Sex'].replace(['[Not Available]','Male','Female'],['00','1','2'],inplace=True)
df1['Race Category'].replace(['[Not Available]','WHITE','BLACK OR AFRICAN AMERICAN','ASIAN','AMERICAN INDIAN OR ALASKA NATIVE'],['00','1','2','3','4'],inplace=True)
df1['Ethnicity Category'].replace(['[Not Available]','NOT HISPANIC OR LATINO','HISPANIC OR LATINO'],['00','1','2'],inplace=True)
df1['Prior Cancer Diagnosis Occurence'].replace(['[Not Available]','No','Yes','Yes, History of Synchronous/Bilateral Malignancy','Yes, History of Prior Malignancy'],['00','1','2','3','4'],inplace=True)
df1['Neoadjuvant Therapy Type Administered Prior To Resection Text'].replace(['[Not Available]','No','Yes'],['00','1','2'],inplace=True)
df1['Vital Status'].replace(['[Not Available]','Dead','Alive'],['00','1','2'],inplace=True)
df1['American Joint Committee on Cancer Publication Version Type'].replace(['[Not Available]','6th','7th','5th','4th'],['00','1','2','3','4'],inplace=True)
df1['American Joint Committee on Cancer Tumor Stage Code'].replace(['[Not Available]','T0','T1','T2','T3','T4','T4a','T4b','TX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Disease Free Status'].replace(['0:DiseaseFree','1:Recurred/Progressed','[Not Available]'],['0','1','00'],inplace=True)
df1['Neoplasm Histologic Grade'].replace(['[Not Available]','G1','G2','G3','GX','G4'],['00','1','2','3','4','5'],inplace=True)
df1['Alcohol History Documented'].replace(['[Not Available]','No','Yes','NO','YES'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','N0','N1','N2','N2a','N2b','N2c','N3','NX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm Disease Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','Discrepancy','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','00','1','2','3','4','5','6'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage'].replace(['[Not Available]','M1','M1','MX','M0'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage'].replace(['[Not Available]','N0','N1','N2a','N2b','N2c','N3','NX','N2'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage'].replace(['[Not Available]','T1','T2','T3','T4a','T4b','TX','T4'],['00','1','2','3','4','5','6','7'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Group Stage'].replace(['[Not Available]','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','1','2','3','4','5','6'],inplace=True)

df1.head()

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,12,1,2,1,1,1,2013,2,2,2,...,66,4,3,4,4,0,0,3.35,0,3.35
TCGA-BA-4074-01,12,1,1,1,1,1,2003,2,1,1,...,69,4,5,3,4,0,1,15.18,1,13.01
TCGA-BA-4075-01,12,1,2,1,2,2,2004,2,1,1,...,49,4,2,4,4,0,1,9.3,1,7.75
TCGA-BA-4076-01,2,1,1,1,1,1,2003,2,1,1,...,39,4,5,3,4,0,1,13.63,1,9.4
TCGA-BA-4077-01,11,2,1,1,2,2,2003,2,1,1,...,45,4,6,5,5,0,1,37.25,1,9.4


In [5]:
#STORING REDUCED DATA TO CSV
cl = pd.DataFrame(df1)
cl.to_csv("CLINICALpreprocessed.csv")
print("Data exported to csv file")

Data exported to csv file


In [6]:
#READING CSV FILE
df =pd.read_csv("data_methylation_hm450.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df2 = df.T
df2.to_csv("Methylation.csv")
df2.head()

Hugo_Symbol,TSEN34,MUSTN1,C3orf16,CKLF,SFRS7,FAM180B,PTPRF,C6orf168,LOC728024,DSTYK,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
TCGA-4P-AA8J-01,0.160658,0.856690,0.689981,0.063268,0.088333,0.738956,0.689032,0.485472,0.847822,0.018144,...,0.021239,0.095778,0.061835,0.043785,0.055341,0.066497,0.080675,0.041548,0.054317,0.098915
TCGA-BA-4074-01,0.172720,0.888797,0.448310,0.095680,0.054274,0.506644,0.842746,0.188788,0.916802,0.018413,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
TCGA-BA-4075-01,0.091838,0.876359,0.336352,0.079018,0.062922,0.475571,0.783786,0.221566,0.792347,0.021204,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
TCGA-BA-4076-01,0.127324,0.911893,0.757925,0.095460,0.073372,0.834641,0.718938,0.791580,0.898727,0.014496,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
TCGA-BA-4077-01,0.132946,0.893790,0.556940,0.074819,0.080927,0.773512,0.392731,0.283078,0.857577,0.018026,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227


In [7]:
# Merge datasets based on the patient identifier
merged_df = pd.merge(cl, df2, left_index=True, right_index=True, how="inner")
df2=merged_df


In [8]:

c2 = pd.DataFrame(df2)
c2.to_csv("Methylationpreprocessed.csv")

In [9]:

# Extract target variable (survival time) from clinical data
y = c2['Overall Survival (Months)']
c2 = c2.drop('Overall Survival (Months)', axis=1)


In [10]:
y.drop(y.index[-1], inplace=True)
dm=c2.iloc[:,:]
#print(d)
dm = dm.reset_index()
M=dm.iloc[1:,1:]
M


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
1,12,1,1,1,1,1,2003,2,1,1,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
2,12,1,2,1,2,2,2004,2,1,1,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
3,2,1,1,1,1,1,2003,2,1,1,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
4,11,2,1,1,2,2,2003,2,1,1,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227
5,2,1,1,1,1,1,2003,2,1,1,...,0.036186,0.065871,0.042553,0.043962,0.036412,0.131708,0.062926,0.043616,0.044998,0.057528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,4,2,1,1,1,1,2009,2,1,1,...,0.019317,0.045418,0.032703,0.032664,0.035815,0.044621,0.080887,0.037113,0.033082,0.048041
524,6,2,1,1,3,1,2011,2,1,2,...,0.030936,0.068638,0.042534,0.043967,0.047700,0.063594,0.094378,0.045045,0.049718,0.084455
525,12,1,1,2,1,1,2013,2,2,2,...,0.024656,0.044608,0.030440,0.047566,0.064576,0.062233,0.051034,0.034986,0.053662,0.051514
526,4,1,1,1,1,1,2012,2,1,2,...,0.026150,0.078658,0.047960,0.048287,0.050811,0.059785,0.077223,0.051496,0.048270,0.059510


In [11]:
encodermyth=Sequential()
encodermyth.add(Dense(units=8000,activation="relu"))
encodermyth.add(Dense(units=4000,activation="relu"))
encodermyth.add(Dense(units=2000,activation="relu"))
encodermyth.add(Dense(units=1000,activation="relu"))
encodermyth.add(Dense(units=500,activation="relu"))
encodermyth.add(Dense(units=280,activation="relu"))

decodermyth=Sequential()
decodermyth.add(Dense(units=500,activation="relu"))
decodermyth.add(Dense(units=1000,activation="relu"))
decodermyth.add(Dense(units=2000,activation="relu"))
decodermyth.add(Dense(units=4000,activation="relu"))
decodermyth.add(Dense(units=8000,activation="relu"))
decodermyth.add(Dense(units=15330,activation="softmax"))
autoencodermyth = Sequential([encodermyth,decodermyth])
autoencodermyth.compile(loss="mse",optimizer=SGD(lr=1.5))
#standardize the data

scaler = StandardScaler()
X1 = scaler.fit_transform(M[:])
#PCA 
# fit pca on data

autoencodermyth.fit(X1,X1,batch_size = 16, shuffle =True, epochs=5)


Z1 = encodermyth.predict(X1)




C:\Users\Pro Deepali\AppData\Roaming\Python\Python39\site-packages\keras\optimizer_v2\gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


Epoch 1/5
33/33 [==============================] - 46s 1s/step - loss: 0.9999
Epoch 2/5
33/33 [==============================] - 42s 1s/step - loss: 0.9999
Epoch 3/5
33/33 [==============================] - 44s 1s/step - loss: 0.9999
Epoch 4/5
33/33 [==============================] - 43s 1s/step - loss: 0.9999
Epoch 5/5
33/33 [==============================] - 44s 1s/step - loss: 0.9999


In [12]:
n1 = pd.DataFrame(Z1)
print(n1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
n1["Hugo"]=hugo
n1

(527, 280)


,0,1,2,3,4,5,6,7,8,9,...,271,272,273,274,275,276,277,278,279,Hugo
0,0.151665,0.000000,0.0,0.0,0.808601,0.000000,0.000000,0.077954,0.0,0.479289,...,0.000000,0.114683,0.0,0.222255,0.427239,0.000000,0.000000,0.858683,0.000000,TCGA-BA-4074-01
1,0.182274,0.359307,0.0,0.0,1.175333,0.812924,0.187238,0.000000,0.0,1.247811,...,0.000000,0.245345,0.0,0.538339,0.221951,0.068159,0.000000,1.820169,0.049397,TCGA-BA-4075-01
2,0.354200,0.000000,0.0,0.0,1.152393,0.571035,0.209389,0.000000,0.0,0.048169,...,0.150274,0.000000,0.0,1.371932,0.414958,0.000000,0.482228,1.135589,0.387501,TCGA-BA-4076-01
3,0.346105,0.000000,0.0,0.0,0.404042,0.455150,0.000000,0.000000,0.0,0.266435,...,0.000000,0.057784,0.0,0.398438,0.147397,0.097758,0.000000,0.712255,0.000000,TCGA-BA-4077-01
4,0.298285,0.000000,0.0,0.0,0.959392,0.000000,0.079514,0.296489,0.0,0.448646,...,0.272984,0.552715,0.0,0.646767,0.230468,0.000000,0.095640,1.203423,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.232106,0.000000,0.0,0.0,0.701408,0.000000,0.000000,0.000000,0.0,0.266727,...,0.000000,0.000000,0.0,0.000000,0.508313,0.000000,0.254180,0.922466,0.000000,TCGA-UF-A7JT-01
523,0.199421,0.000000,0.0,0.0,0.546006,0.000000,0.237784,0.062844,0.0,0.465966,...,0.000000,0.000000,0.0,0.098787,0.028698,0.000000,0.000000,0.686134,0.000000,TCGA-UF-A7JV-01
524,0.483773,0.067931,0.0,0.0,1.346014,0.000000,0.000000,0.000000,0.0,0.486659,...,0.060849,0.000000,0.0,1.105598,0.001861,0.000000,0.271835,1.741808,0.000000,TCGA-UP-A6WW-01
525,0.453002,0.000000,0.0,0.0,1.764444,0.000000,0.000000,0.737510,0.0,0.869799,...,0.000000,0.125628,0.0,0.847956,0.786085,0.000000,0.000000,1.638911,0.000000,TCGA-WA-A7GZ-01


In [13]:

lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z1, y)
n_components = Z1.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(168).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zl1 = np.column_stack(columns)
nl1 = pd.DataFrame(Zl1)
print(nl1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nl1["Hugo"]=hugo
nl1

Number of selected features:  168
Selected features:  Int64Index([169,  76, 168,   5, 203, 163, 205, 238, 111,   2,
            ...
            125, 126, 127, 128, 129, 131, 132, 133, 134, 135],
           dtype='int64', length=168)
(527, 168)


,0,1,2,3,4,5,6,7,8,9,...,159,160,161,162,163,164,165,166,167,Hugo
0,0.000000,0.053536,0.318962,0.000000,0.179673,0.059106,0.000000,0.089996,0.014290,0.0,...,0.0,0.429020,0.0,0.000000,0.000000,0.000000,0.118988,0.0,0.0,TCGA-BA-4074-01
1,0.000000,0.545789,0.000000,0.812924,0.138904,0.000000,0.000000,0.746501,0.000000,0.0,...,0.0,1.819256,0.0,0.191585,0.284449,0.000000,0.000000,0.0,0.0,TCGA-BA-4075-01
2,0.129284,0.136444,0.330828,0.571035,0.225088,0.000000,0.099241,0.215831,0.000000,0.0,...,0.0,0.618193,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,TCGA-BA-4076-01
3,0.000000,0.227871,0.289890,0.455150,0.000000,0.000000,0.492085,0.089968,0.000000,0.0,...,0.0,0.404105,0.0,0.030776,0.040857,0.000000,0.000000,0.0,0.0,TCGA-BA-4077-01
4,0.223388,0.415408,0.107307,0.000000,0.227111,0.000000,0.103781,0.525226,0.226709,0.0,...,0.0,1.124045,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.000000,0.000000,0.000000,0.000000,0.040015,0.000000,0.307902,0.425395,0.000000,0.0,...,0.0,0.607785,0.0,0.109356,0.187567,0.082103,0.000000,0.0,0.0,TCGA-UF-A7JT-01
523,0.114678,0.000000,0.055122,0.000000,0.000000,0.086714,0.056974,0.000000,0.000000,0.0,...,0.0,0.792811,0.0,0.000000,0.000000,0.000000,0.126692,0.0,0.0,TCGA-UF-A7JV-01
524,0.394292,0.000000,0.110302,0.000000,0.000000,0.035536,0.444956,0.422808,0.024785,0.0,...,0.0,0.822482,0.0,0.278728,0.000000,0.000000,0.000000,0.0,0.0,TCGA-UP-A6WW-01
525,0.414611,0.123185,0.000000,0.000000,0.648738,0.341867,0.466876,0.058997,0.020222,0.0,...,0.0,1.335442,0.0,0.068650,0.000000,0.000000,0.000000,0.0,0.0,TCGA-WA-A7GZ-01


In [14]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z1, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z1.shape[1])])
selected_features = coef.abs().nlargest(168).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Ze1 = np.column_stack(columns)
ne1 = pd.DataFrame(Ze1)
ne1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
ne1["Hugo"]=hugo
ne1


Number of selected features:  168
Selected features:  [163, 5, 177, 117, 243, 169, 35, 227, 118, 203, 131, 10, 257, 123, 89, 139, 238, 164, 174, 130, 279, 205, 1, 2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 119, 120, 121, 122, 124, 125, 126, 127, 128, 129, 132, 133, 134, 135, 136, 137, 138, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156]


,0,1,2,3,4,5,6,7,8,9,...,159,160,161,162,163,164,165,166,167,Hugo
0,0.059106,0.000000,0.000000,0.362197,0.354257,0.000000,0.446995,0.648103,0.232511,0.179673,...,0.467691,0.238509,0.000000,0.007789,0.0,0.0,0.165505,0.036238,0.0,TCGA-BA-4074-01
1,0.000000,0.812924,0.000000,0.543185,0.490616,0.000000,0.870916,1.294451,0.138400,0.138904,...,0.403322,1.148387,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,TCGA-BA-4075-01
2,0.000000,0.571035,0.112559,0.000000,0.000000,0.129284,0.443902,0.818035,0.365188,0.225088,...,0.695199,0.699306,0.346178,0.000000,0.0,0.0,0.000000,0.000000,0.0,TCGA-BA-4076-01
3,0.000000,0.455150,0.000000,0.285106,0.011693,0.000000,0.000000,0.249499,0.077728,0.000000,...,0.230313,0.313482,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,TCGA-BA-4077-01
4,0.000000,0.000000,0.000000,0.000000,0.066420,0.223388,0.000000,0.263781,0.000000,0.227111,...,0.003822,0.665762,0.300544,0.000000,0.0,0.0,0.048512,0.000000,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.000000,0.000000,0.000000,0.049027,0.025456,0.000000,0.472419,0.752054,0.000000,0.040015,...,0.145365,0.401158,0.000000,0.148940,0.0,0.0,0.076153,0.000000,0.0,TCGA-UF-A7JT-01
523,0.086714,0.000000,0.000000,0.327891,0.109133,0.114678,0.457074,0.492376,0.457025,0.000000,...,0.348215,0.328936,0.002410,0.000000,0.0,0.0,0.000000,0.263056,0.0,TCGA-UF-A7JV-01
524,0.035536,0.000000,0.086975,0.155641,0.720044,0.394292,0.748997,0.743863,0.097007,0.000000,...,0.767819,0.904638,0.000000,0.000000,0.0,0.0,0.247718,0.000000,0.0,TCGA-UP-A6WW-01
525,0.341867,0.000000,0.000000,0.809473,0.957196,0.414611,0.336243,1.821074,0.000000,0.648738,...,0.402221,1.396735,0.000000,0.029027,0.0,0.0,0.000000,0.000000,0.0,TCGA-WA-A7GZ-01


In [15]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z1, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z1.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(168)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zrf1 = np.column_stack(columns)
nrf1 = pd.DataFrame(Zrf1)
nrf1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nrf1["Hugo"]=hugo
nrf1

Number of selected features:  168
Selected features:  [168, 164, 227, 161, 242, 35, 228, 113, 208, 51, 15, 267, 88, 5, 60, 76, 243, 55, 209, 150, 174, 45, 117, 223, 245, 89, 256, 219, 34, 24, 128, 49, 250, 151, 231, 163, 102, 202, 244, 204, 66, 177, 39, 38, 80, 140, 78, 229, 138, 169, 131, 234, 105, 9, 41, 61, 63, 26, 82, 40, 71, 77, 21, 203, 239, 143, 201, 91, 173, 19, 20, 57, 199, 115, 103, 17, 1, 16, 139, 180, 123, 101, 94, 200, 87, 257, 226, 124, 36, 31, 215, 175, 258, 33, 278, 148, 159, 100, 190, 23, 12, 6, 46, 86, 222, 72, 56, 249, 218, 241, 10, 189, 13, 276, 145, 263, 50, 126, 198, 170, 130, 18, 224, 165, 122, 206, 114, 90, 160, 235, 236, 217, 193, 262, 172, 265, 149, 261, 213, 152, 147, 104, 120, 195, 181, 107, 70, 84, 205, 54, 111, 176, 275, 260, 110, 184, 268, 47, 74, 214, 42, 221, 73, 142, 253, 272, 81, 27]


,0,1,2,3,4,5,6,7,8,9,...,159,160,161,162,163,164,165,166,167,Hugo
0,0.318962,0.057620,0.648103,0.0,0.529884,0.446995,0.000000,0.510013,0.000000,0.0,...,0.000000,0.0,0.428346,0.625257,0.849398,0.000000,0.114683,0.136431,0.0,TCGA-BA-4074-01
1,0.000000,0.768572,1.294451,0.0,0.797252,0.870916,0.000000,0.545356,0.000000,0.0,...,1.126109,0.0,0.000000,1.279058,1.558474,0.000000,0.245345,0.000000,0.0,TCGA-BA-4075-01
2,0.330828,0.038378,0.818035,0.0,0.360999,0.443902,0.000000,0.343623,0.000000,0.0,...,0.000000,0.0,0.491034,0.924320,0.850549,0.000000,0.000000,0.336126,0.0,TCGA-BA-4076-01
3,0.289890,0.069690,0.249499,0.0,0.000000,0.000000,0.000000,0.303615,0.142095,0.0,...,0.000000,0.0,0.026314,0.739170,0.596105,0.000000,0.057784,0.355425,0.0,TCGA-BA-4077-01
4,0.107307,0.399196,0.263781,0.0,0.579603,0.000000,0.173583,0.441881,0.000000,0.0,...,0.030216,0.0,0.005881,1.086318,0.844522,0.000000,0.552715,0.370107,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.000000,0.351317,0.752054,0.0,0.248345,0.472419,0.183570,0.319139,0.000000,0.0,...,0.396177,0.0,0.007181,0.578908,0.708737,0.000000,0.000000,0.327808,0.0,TCGA-UF-A7JT-01
523,0.055122,0.051257,0.492376,0.0,0.213727,0.457074,0.000000,0.699633,0.059944,0.0,...,0.276719,0.0,0.425729,0.733177,0.700937,0.000000,0.000000,0.200281,0.0,TCGA-UF-A7JV-01
524,0.110302,0.029778,0.743863,0.0,0.208430,0.748997,0.000000,0.718040,0.075973,0.0,...,0.082361,0.0,0.033706,0.253917,0.814862,0.000000,0.000000,0.291747,0.0,TCGA-UP-A6WW-01
525,0.000000,0.000000,1.821074,0.0,0.019513,0.336243,0.392271,0.788719,0.306515,0.0,...,0.214823,0.0,1.194353,1.908905,1.398079,0.000000,0.125628,0.443043,0.0,TCGA-WA-A7GZ-01


In [16]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=168)
rfe.fit(Z1, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zre1 = np.column_stack(columns)
nre1 = pd.DataFrame(Zre1)
nre1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nre1["Hugo"]=hugo
nre1

Selected Features: [2, 3, 5, 8, 9, 10, 11, 12, 15, 16, 17, 19, 20, 22, 24, 26, 27, 29, 30, 32, 33, 35, 36, 37, 38, 41, 42, 46, 47, 49, 50, 51, 53, 54, 57, 58, 61, 62, 63, 65, 66, 69, 70, 71, 72, 75, 76, 80, 81, 83, 84, 85, 86, 87, 89, 91, 92, 93, 94, 95, 97, 98, 99, 102, 104, 106, 108, 110, 111, 112, 117, 118, 119, 120, 121, 122, 123, 124, 126, 127, 128, 129, 130, 132, 133, 134, 135, 137, 139, 141, 144, 145, 148, 151, 154, 155, 156, 157, 158, 160, 161, 162, 163, 165, 168, 177, 178, 179, 183, 184, 187, 188, 190, 191, 192, 194, 195, 196, 197, 198, 200, 202, 203, 205, 206, 207, 211, 212, 215, 216, 218, 220, 221, 222, 223, 225, 226, 231, 232, 233, 238, 239, 240, 242, 243, 244, 245, 248, 249, 252, 253, 254, 255, 256, 257, 258, 259, 260, 262, 264, 266, 269, 271, 275, 276, 277, 278, 279]


,0,1,2,3,4,5,6,7,8,9,...,159,160,161,162,163,164,165,166,167,Hugo
0,0.0,0.0,0.000000,0.0,0.479289,0.000000,0.000000,0.000000,0.395993,0.307739,...,0.341480,0.281882,0.0,0.000000,0.427239,0.000000,0.000000,0.858683,0.000000,TCGA-BA-4074-01
1,0.0,0.0,0.812924,0.0,1.247811,0.000000,0.998308,0.000000,0.159242,0.000000,...,0.163497,0.967816,0.0,0.000000,0.221951,0.068159,0.000000,1.820169,0.049397,TCGA-BA-4075-01
2,0.0,0.0,0.571035,0.0,0.048169,0.307197,0.143627,0.300028,0.113454,0.855996,...,0.128628,0.540298,0.0,0.150274,0.414958,0.000000,0.482228,1.135589,0.387501,TCGA-BA-4076-01
3,0.0,0.0,0.455150,0.0,0.266435,0.000000,0.380179,0.000000,0.000000,0.009655,...,0.041980,0.529833,0.0,0.000000,0.147397,0.097758,0.000000,0.712255,0.000000,TCGA-BA-4077-01
4,0.0,0.0,0.000000,0.0,0.448646,0.000000,0.379055,0.000000,0.005735,0.484960,...,0.814716,0.426829,0.0,0.272984,0.230468,0.000000,0.095640,1.203423,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.0,0.0,0.000000,0.0,0.266727,0.000000,0.366971,0.000000,0.046123,0.399586,...,0.000000,0.505920,0.0,0.000000,0.508313,0.000000,0.254180,0.922466,0.000000,TCGA-UF-A7JT-01
523,0.0,0.0,0.000000,0.0,0.465966,0.000000,0.000000,0.103489,0.279050,0.364774,...,0.271985,0.497745,0.0,0.000000,0.028698,0.000000,0.000000,0.686134,0.000000,TCGA-UF-A7JV-01
524,0.0,0.0,0.000000,0.0,0.486659,0.000000,0.295857,0.000000,0.233949,0.847855,...,0.000000,0.135269,0.0,0.060849,0.001861,0.000000,0.271835,1.741808,0.000000,TCGA-UP-A6WW-01
525,0.0,0.0,0.000000,0.0,0.869799,0.000000,0.259575,0.000000,0.751374,0.524055,...,0.379473,0.315986,0.0,0.000000,0.786085,0.000000,0.000000,1.638911,0.000000,TCGA-WA-A7GZ-01


In [17]:
#READING CSV FILE
df = pd.read_csv("data_RNA_Seq_v2_expression_median.csv")

#DROP ROWS AND COLUMNS
df=df.replace(0,np.nan)
df=df.dropna()
df=df.replace(np.nan,0)
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df4 = df.T
df4.to_csv("RNAb.csv")
df4.head()
merged_df = pd.merge(cl, df4, left_index=True, right_index=True, how="inner")
df4=merged_df

In [18]:
c4 = pd.DataFrame(df4)
c4.to_csv("RNApreprocessed.csv")
y = c4['Overall Survival (Months)']
c4 = c4.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
drn=c4.iloc[:,:]
#print(d)
drn = drn.reset_index()
Rn=drn.iloc[1:,1:]
Rn.head()


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,LOC154274,ZW10,ZWILCH,ZWINT,ZXDB,LOC100130182,ZYG11B,ZYX,FLJ10821,ZZZ3
1,12,1,1,1,1,1,2003,2,1,1,...,311.0030,283.6409,2132.4595,1193.1417,172.2656,380.3717,805.0624,2516.9279,258.5911,1088.3179
2,12,1,2,1,2,2,2004,2,1,1,...,225.1105,512.3945,761.0023,673.1877,172.0488,562.2404,487.7395,5930.0549,292.6437,980.3028
3,2,1,1,1,1,1,2003,2,1,1,...,157.9431,307.4905,480.0682,1032.6643,324.2818,1440.9025,722.5502,2674.5376,672.1763,998.5570
4,11,2,1,1,2,2,2003,2,1,1,...,137.6323,361.4052,1325.3128,1620.3080,210.7796,1423.0029,770.9336,8035.6112,763.2339,692.9740
5,2,1,1,1,1,1,2003,2,1,1,...,241.8520,414.1231,874.1257,1145.1112,372.9953,2634.2473,780.1345,3895.2406,1556.6477,1309.6223


Applying PCA dimensioality reduction technique

In [19]:
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X3 = scaler.fit_transform(Rn[:])
#fit pca on data
encodermrna=Sequential()
encodermrna.add(Dense(units=12000,activation="relu"))
encodermrna.add(Dense(units=6000,activation="relu"))
encodermrna.add(Dense(units=3000,activation="relu"))
encodermrna.add(Dense(units=1500,activation="relu"))
encodermrna.add(Dense(units=750,activation="relu"))
encodermrna.add(Dense(units=300,activation="relu"))

decodermrna=Sequential()
decodermrna.add(Dense(units=750,activation="relu"))
decodermrna.add(Dense(units=1500,activation="relu"))
decodermrna.add(Dense(units=3000,activation="relu"))
decodermrna.add(Dense(units=6000,activation="relu"))
decodermrna.add(Dense(units=12000,activation="relu"))
decodermrna.add(Dense(units=12723,activation="relu"))
autoencodermrna = Sequential([encodermrna,decodermrna])
autoencodermrna.compile(loss="mse",optimizer=SGD(lr=1.5))
autoencodermrna.fit(X3,X3,batch_size = 16, shuffle =True, epochs=4)
Z3 = encodermrna.predict(X3)


Epoch 1/4


C:\Users\Pro Deepali\AppData\Roaming\Python\Python39\site-packages\keras\optimizer_v2\gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


33/33 [==============================] - 73s 2s/step - loss: 0.9988
Epoch 2/4
33/33 [==============================] - 62s 2s/step - loss: 0.9763
Epoch 3/4
33/33 [==============================] - 63s 2s/step - loss: 0.9971
Epoch 4/4
33/33 [==============================] - 62s 2s/step - loss: 0.9740


Applying RFE Feature selection methods

In [20]:
n3 = pd.DataFrame(Z3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
n3["Hugo"]=hugo
n3

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,Hugo
0,10.643723,7.563281,3.948407,0.000000,3.067497,4.665798,0.0,11.991080,7.111972,0.000000,...,7.457398,0.000000,4.679759,1.448159,0.0,2.062253,2.326864,2.104791,0.0,TCGA-BA-4074-01
1,6.861281,4.624980,2.127479,0.000000,1.586897,2.299753,0.0,7.731060,4.796114,0.000000,...,4.987992,0.000000,2.778082,0.883089,0.0,1.547343,1.366306,1.157931,0.0,TCGA-BA-4075-01
2,0.000000,0.387210,0.463572,0.000000,0.099206,0.000000,0.0,0.112800,0.526848,0.209059,...,0.464226,0.000000,0.425002,0.000000,0.0,0.055066,0.000000,0.053725,0.0,TCGA-BA-4076-01
3,0.000000,0.001714,0.163877,0.055459,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.008963,0.000000,0.000000,0.000000,0.0,0.181391,0.000000,0.000000,0.0,TCGA-BA-4077-01
4,0.282393,0.872086,0.989336,0.327366,0.206057,0.135779,0.0,0.470460,0.272756,0.204796,...,0.827472,0.155287,0.223604,0.003464,0.0,0.000000,0.000000,0.337018,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,5.074081,3.003355,1.278715,0.000000,0.835270,1.543512,0.0,5.669614,3.294031,0.000000,...,3.451087,0.000000,1.759115,0.479295,0.0,0.974961,0.575060,1.019199,0.0,TCGA-UF-A7JT-01
515,0.838433,0.963502,0.601659,0.000000,0.000000,0.355071,0.0,0.954360,1.009169,0.000000,...,1.107270,0.000000,0.485680,0.073259,0.0,0.091941,0.000000,0.000000,0.0,TCGA-UF-A7JV-01
516,8.420542,5.451691,2.240130,0.000000,1.697917,3.221523,0.0,9.326238,5.640399,0.000000,...,5.535241,0.000000,3.586300,1.143958,0.0,2.106303,1.502867,1.994194,0.0,TCGA-UP-A6WW-01
517,0.669475,0.353222,0.371013,0.000000,0.000000,0.224225,0.0,1.154265,1.158021,0.000000,...,0.880826,0.000000,0.443120,0.000000,0.0,0.045379,0.090377,0.196469,0.0,TCGA-WA-A7GZ-01


In [21]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z3, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z3.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(180)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zrf3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf3 = pd.DataFrame(Zrf3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nrf3["Hugo"]=hugo
nrf3

Number of selected features:  180
Selected features:  [100, 151, 173, 200, 146, 264, 160, 187, 203, 241, 225, 113, 77, 108, 188, 163, 73, 180, 157, 277, 227, 101, 53, 144, 50, 245, 182, 140, 81, 185, 283, 128, 137, 25, 243, 248, 107, 76, 56, 2, 159, 131, 289, 30, 170, 39, 216, 290, 212, 114, 126, 152, 3, 109, 184, 220, 284, 88, 86, 162, 5, 66, 233, 28, 20, 269, 71, 298, 167, 120, 222, 204, 51, 190, 46, 19, 29, 119, 47, 209, 98, 69, 85, 121, 286, 291, 93, 127, 165, 23, 32, 22, 191, 158, 192, 90, 282, 236, 178, 270, 201, 24, 149, 224, 280, 207, 141, 244, 176, 242, 124, 213, 70, 232, 255, 45, 55, 49, 218, 134, 293, 54, 223, 221, 122, 33, 21, 44, 268, 52, 198, 91, 276, 89, 60, 254, 206, 17, 65, 57, 27, 197, 136, 208, 257, 258, 239, 145, 153, 82, 142, 40, 215, 4, 249, 278, 97, 166, 230, 274, 196, 26, 58, 139, 125, 34, 279, 266, 297, 234, 87, 261, 12, 299, 265, 189, 183, 194, 6, 10]


,0,1,2,3,4,5,6,7,8,9,...,171,172,173,174,175,176,177,178,179,Hugo
0,0.535108,0.0,0.0,0.0,0.0,9.521526,0.0,0.000000,0.0,6.488761,...,0.0,0.0,0.0,0.0,1.263038,1.550415,0.0,0.0,7.655177,TCGA-BA-4074-01
1,0.838156,0.0,0.0,0.0,0.0,5.973319,0.0,0.107849,0.0,3.874504,...,0.0,0.0,0.0,0.0,0.289566,0.413215,0.0,0.0,5.145136,TCGA-BA-4075-01
2,0.000000,0.0,0.0,0.0,0.0,0.523323,0.0,0.000000,0.0,0.579055,...,0.0,0.0,0.0,0.0,0.249600,0.000000,0.0,0.0,0.004826,TCGA-BA-4076-01
3,0.000000,0.0,0.0,0.0,0.0,0.185988,0.0,0.000000,0.0,0.136336,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,TCGA-BA-4077-01
4,0.569237,0.0,0.0,0.0,0.0,1.103148,0.0,0.000000,0.0,1.122817,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.065699,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.936758,0.0,0.0,0.0,0.0,3.739181,0.0,0.090314,0.0,2.866604,...,0.0,0.0,0.0,0.0,0.000000,0.699137,0.0,0.0,3.679617,TCGA-UF-A7JT-01
515,0.433511,0.0,0.0,0.0,0.0,1.149104,0.0,0.000000,0.0,0.923808,...,0.0,0.0,0.0,0.0,0.000000,0.119847,0.0,0.0,0.494282,TCGA-UF-A7JV-01
516,1.781407,0.0,0.0,0.0,0.0,7.294072,0.0,0.000000,0.0,5.235215,...,0.0,0.0,0.0,0.0,0.000000,0.837403,0.0,0.0,6.584057,TCGA-UP-A6WW-01
517,0.060337,0.0,0.0,0.0,0.0,0.990967,0.0,0.000000,0.0,0.973708,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.813522,TCGA-WA-A7GZ-01


In [22]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=180)
rfe.fit(Z3, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zre3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre3 = pd.DataFrame(Zre3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nre3["Hugo"]=hugo
nre3

Selected Features: [2, 3, 4, 5, 9, 10, 12, 15, 17, 18, 19, 21, 22, 23, 24, 26, 27, 29, 31, 33, 34, 35, 39, 40, 41, 42, 43, 44, 47, 48, 49, 51, 52, 53, 55, 56, 57, 61, 64, 69, 70, 72, 76, 77, 78, 79, 81, 82, 83, 84, 85, 86, 88, 89, 90, 91, 93, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 108, 109, 110, 113, 117, 118, 119, 120, 122, 123, 125, 126, 127, 131, 135, 136, 137, 139, 140, 141, 142, 144, 147, 150, 151, 152, 154, 155, 156, 157, 159, 160, 164, 168, 169, 171, 172, 173, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 187, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 200, 202, 203, 207, 208, 210, 212, 216, 218, 220, 222, 230, 232, 233, 234, 239, 241, 243, 244, 246, 248, 249, 250, 255, 256, 257, 259, 260, 261, 263, 264, 265, 266, 270, 272, 274, 278, 279, 280, 281, 285, 286, 287, 288, 291, 292, 294, 295, 296, 298, 299]


,0,1,2,3,4,5,6,7,8,9,...,171,172,173,174,175,176,177,178,179,Hugo
0,3.948407,0.000000,3.067497,4.665798,0.000000,7.655177,0.0,0.0,3.238794,1.428862,...,2.887454,1.030205,7.457398,0.000000,1.448159,0.0,2.062253,2.104791,0.0,TCGA-BA-4074-01
1,2.127479,0.000000,1.586897,2.299753,0.000000,5.145136,0.0,0.0,2.057744,0.876823,...,1.214130,0.511622,4.987992,0.000000,0.883089,0.0,1.547343,1.157931,0.0,TCGA-BA-4075-01
2,0.463572,0.000000,0.099206,0.000000,0.209059,0.004826,0.0,0.0,0.000000,0.000000,...,0.000849,0.421134,0.464226,0.000000,0.000000,0.0,0.055066,0.053725,0.0,TCGA-BA-4076-01
3,0.163877,0.055459,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.153227,0.008963,0.000000,0.000000,0.0,0.181391,0.000000,0.0,TCGA-BA-4077-01
4,0.989336,0.327366,0.206057,0.135779,0.204796,0.065699,0.0,0.0,0.000000,0.000000,...,0.068540,0.314266,0.827472,0.155287,0.003464,0.0,0.000000,0.337018,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,1.278715,0.000000,0.835270,1.543512,0.000000,3.679617,0.0,0.0,0.809844,0.679333,...,0.550774,0.000000,3.451087,0.000000,0.479295,0.0,0.974961,1.019199,0.0,TCGA-UF-A7JT-01
515,0.601659,0.000000,0.000000,0.355071,0.000000,0.494282,0.0,0.0,0.000000,0.098145,...,0.000000,0.103222,1.107270,0.000000,0.073259,0.0,0.091941,0.000000,0.0,TCGA-UF-A7JV-01
516,2.240130,0.000000,1.697917,3.221523,0.000000,6.584057,0.0,0.0,2.328816,1.408352,...,1.728235,0.782908,5.535241,0.000000,1.143958,0.0,2.106303,1.994194,0.0,TCGA-UP-A6WW-01
517,0.371013,0.000000,0.000000,0.224225,0.000000,0.813522,0.0,0.0,0.250987,0.000000,...,0.167766,0.195376,0.880826,0.000000,0.000000,0.0,0.045379,0.196469,0.0,TCGA-WA-A7GZ-01


In [23]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z3, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z3.shape[1])])
selected_features = coef.abs().nlargest(180).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Ze3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne3 = pd.DataFrame(Ze3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
ne3["Hugo"]=hugo
ne3


G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 199.0331573486328, tolerance: 37.63673400878906
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 93.30339813232422, tolerance: 37.63673400878906
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 205.34376525878906, tolerance: 37.63673400878906
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increas

G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 5502.310546875, tolerance: 38.64818572998047
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 28.40516471862793, tolerance: 23.803607940673828
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 154.476806640625, tolerance: 23.803607940673828
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase t

G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 2350.281005859375, tolerance: 29.325565338134766
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 3222.231201171875, tolerance: 29.325565338134766
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 4248.83447265625, tolerance: 29.325565338134766
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increa

Number of selected features:  180
Selected features:  [130, 272, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179]


,0,1,2,3,4,5,6,7,8,9,...,171,172,173,174,175,176,177,178,179,Hugo
0,0.000000,0.0,7.563281,3.948407,0.000000,3.067497,4.665798,0.0,11.991080,7.111972,...,0.0,0.0,0.0,0.0,3.763036,0.0,0.211167,0.0,1.219641,TCGA-BA-4074-01
1,0.000000,0.0,4.624980,2.127479,0.000000,1.586897,2.299753,0.0,7.731060,4.796114,...,0.0,0.0,0.0,0.0,2.119291,0.0,0.369011,0.0,0.441331,TCGA-BA-4075-01
2,0.000000,0.0,0.387210,0.463572,0.000000,0.099206,0.000000,0.0,0.112800,0.526848,...,0.0,0.0,0.0,0.0,0.102011,0.0,0.000000,0.0,0.000000,TCGA-BA-4076-01
3,0.116587,0.0,0.001714,0.163877,0.055459,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.047580,0.0,0.000000,0.0,0.061435,TCGA-BA-4077-01
4,0.224944,0.0,0.872086,0.989336,0.327366,0.206057,0.135779,0.0,0.470460,0.272756,...,0.0,0.0,0.0,0.0,0.467779,0.0,0.276822,0.0,0.179645,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.000000,0.0,3.003355,1.278715,0.000000,0.835270,1.543512,0.0,5.669614,3.294031,...,0.0,0.0,0.0,0.0,1.808433,0.0,0.063318,0.0,0.812091,TCGA-UF-A7JT-01
515,0.000000,0.0,0.963502,0.601659,0.000000,0.000000,0.355071,0.0,0.954360,1.009169,...,0.0,0.0,0.0,0.0,0.410400,0.0,0.032387,0.0,0.000000,TCGA-UF-A7JV-01
516,0.000000,0.0,5.451691,2.240130,0.000000,1.697917,3.221523,0.0,9.326238,5.640399,...,0.0,0.0,0.0,0.0,3.080392,0.0,0.461969,0.0,1.546011,TCGA-UP-A6WW-01
517,0.199445,0.0,0.353222,0.371013,0.000000,0.000000,0.224225,0.0,1.154265,1.158021,...,0.0,0.0,0.0,0.0,0.472321,0.0,0.082623,0.0,0.403476,TCGA-WA-A7GZ-01


In [24]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z3, y)
n_components = Z3.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(180).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zl3 = np.column_stack(columns)
nl3 = pd.DataFrame(Zl3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nl3["Hugo"]=hugo
nl3

G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.225e+04, tolerance: 4.215e+01
  model = cd_fast.enet_coordinate_descent(


Number of selected features:  180
Selected features:  Int64Index([257,   9,  24, 291,  69, 159, 120, 126, 168,  97,
            ...
            150, 151, 152, 153, 154, 155, 156, 157, 158, 161],
           dtype='int64', length=180)


,0,1,2,3,4,5,6,7,8,9,...,171,172,173,174,175,176,177,178,179,Hugo
0,2.038810,0.000000,0.891073,7.457398,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,16.153656,0.0,0.000000,0.0,0.000000,0.0,4.261504,0.000000,TCGA-BA-4074-01
1,1.816843,0.000000,0.414497,4.987992,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,10.400257,0.0,0.000000,0.0,0.000000,0.0,2.307795,0.000000,TCGA-BA-4075-01
2,0.517484,0.209059,0.000000,0.464226,0.000000,0.004055,0.035450,0.002934,0.0,0.0,...,0.0,0.493472,0.0,0.000000,0.0,0.000000,0.0,0.190650,0.208077,TCGA-BA-4076-01
3,0.077384,0.000000,0.000000,0.008963,0.000000,0.111132,0.042607,0.035666,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.069920,0.0,0.000000,0.188186,TCGA-BA-4077-01
4,0.209869,0.204796,0.035200,0.827472,0.010622,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.588526,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,1.271710,0.000000,0.380580,3.451087,0.000000,0.136156,0.000000,0.000000,0.0,0.0,...,0.0,7.581449,0.0,0.000000,0.0,0.000000,0.0,2.074518,0.000000,TCGA-UF-A7JT-01
515,0.529944,0.000000,0.000000,1.107270,0.000000,0.049729,0.000000,0.000000,0.0,0.0,...,0.0,1.785970,0.0,0.000000,0.0,0.000000,0.0,0.555108,0.000000,TCGA-UF-A7JV-01
516,2.513750,0.000000,0.586763,5.535241,0.000000,0.146085,0.000000,0.000000,0.0,0.0,...,0.0,12.347768,0.0,0.000000,0.0,0.000000,0.0,3.210995,0.000000,TCGA-UP-A6WW-01
517,0.304055,0.000000,0.000000,0.880826,0.000000,0.036549,0.000000,0.000000,0.0,0.0,...,0.0,1.543476,0.0,0.000000,0.0,0.000000,0.0,0.130002,0.000000,TCGA-WA-A7GZ-01


In [25]:
df = pd.read_csv("data_linear_CNA.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df5 = df.T
df5.to_csv("CNA.csv")
df5.head()
merged_df = pd.merge(cl, df5, left_index=True, right_index=True, how="inner")
df5=merged_df
c5 = pd.DataFrame(df5)
c5.to_csv("CNApreprocessed.csv")
y = c5['Overall Survival (Months)']
c5 = c5.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
dcn=c5.iloc[:,:]
#print(d)
dcn = dcn.reset_index()
Cn=dcn.iloc[1:,1:]
Cn.head()
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X4 = scaler.fit_transform(Cn[:])
encodercna=Sequential()
encodercna.add(Dense(units=12000,activation="relu"))
encodercna.add(Dense(units=6000,activation="relu"))
encodercna.add(Dense(units=3000,activation="relu"))
encodercna.add(Dense(units=1500,activation="relu"))
encodercna.add(Dense(units=750,activation="relu"))
encodercna.add(Dense(units=500,activation="relu"))
encodercna.add(Dense(units=300,activation="relu"))

decodercna=Sequential()
decodercna.add(Dense(units=500,activation="relu"))
decodercna.add(Dense(units=750,activation="relu"))
decodercna.add(Dense(units=1500,activation="relu"))
decodercna.add(Dense(units=3000,activation="relu"))
decodercna.add(Dense(units=6000,activation="relu"))
decodercna.add(Dense(units=12000,activation="relu"))
decodercna.add(Dense(units=23311,activation="relu"))
autoencodercna = Sequential([encodercna,decodercna])
autoencodercna.compile(loss="mse",optimizer=SGD(lr=1.5))
autoencodercna.fit(X4,X4,epochs=5,batch_size = 16, shuffle =True)
Z4 = encodercna.predict(X4)

n4 = pd.DataFrame(Z4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
n4["Hugo"]=hugo
n4

C:\Users\Pro Deepali\AppData\Roaming\Python\Python39\site-packages\keras\optimizer_v2\gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


Epoch 1/5
33/33 [==============================] - 125s 3s/step - loss: 0.9984
Epoch 2/5
33/33 [==============================] - 97s 3s/step - loss: 0.9839
Epoch 3/5
33/33 [==============================] - 83s 2s/step - loss: 0.9824
Epoch 4/5
33/33 [==============================] - 84s 3s/step - loss: 0.9830
Epoch 5/5
33/33 [==============================] - 86s 3s/step - loss: 0.9999


,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,Hugo
0,0.000000,0.081172,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.148655,0.007795,...,0.0,0.102404,0.0,0.171080,0.065647,0.000000,0.000000,0.044346,0.000000,TCGA-BA-4074-01
1,0.106497,0.000000,0.0,0.000000,0.020181,0.074962,0.0,0.005390,0.000000,0.028067,...,0.0,0.023729,0.0,0.374158,0.437570,0.074314,0.000000,0.000000,0.011279,TCGA-BA-4075-01
2,0.000000,0.471685,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.128463,0.000000,...,0.0,0.084648,0.0,0.495610,0.261970,0.000000,0.048809,0.192494,0.000000,TCGA-BA-4076-01
3,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.199772,0.196698,0.042865,...,0.0,0.000000,0.0,0.201181,0.172830,0.000000,0.000000,0.000000,0.000000,TCGA-BA-4077-01
4,0.000000,0.172377,0.0,0.173410,0.000000,0.014258,0.0,0.000000,0.061938,0.095455,...,0.0,0.000000,0.0,0.337252,0.002249,0.005172,0.000000,0.152827,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,0.000000,0.065872,0.0,0.013718,0.000000,0.000000,0.0,0.000000,0.078632,0.000000,...,0.0,0.034893,0.0,0.147555,0.000000,0.000000,0.000000,0.000000,0.000000,TCGA-UF-A7JT-01
517,0.000000,0.051149,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.123591,0.006805,...,0.0,0.053947,0.0,0.090203,0.099151,0.008361,0.014566,0.016773,0.000000,TCGA-UF-A7JV-01
518,0.000000,0.082600,0.0,0.000000,0.000000,0.136186,0.0,0.116810,0.033647,0.189591,...,0.0,0.188678,0.0,0.072162,0.076295,0.000000,0.036101,0.150455,0.000000,TCGA-UP-A6WW-01
519,0.000000,0.192684,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.081115,0.0,0.092812,0.184369,0.000000,0.000000,0.101356,0.000000,TCGA-WA-A7GZ-01


In [26]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z4, y)
n_components = Z4.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(180).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zl4 = np.column_stack(columns)
nl4 = pd.DataFrame(Zl4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nl4["Hugo"]=hugo
nl4

Number of selected features:  180
Selected features:  Int64Index([ 22,  94, 268, 163, 171, 119, 295, 245, 246, 273,
            ...
            153, 154, 157, 158, 159, 160, 161, 162, 164, 165],
           dtype='int64', length=180)


,0,1,2,3,4,5,6,7,8,9,...,171,172,173,174,175,176,177,178,179,Hugo
0,0.099348,0.000000,0.054465,0.023294,0.000000,0.0,0.065647,0.021662,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.156545,0.00000,0.012698,0.000000,0.020079,TCGA-BA-4074-01
1,0.000000,0.000000,0.540721,0.167058,0.000000,0.0,0.437570,0.206068,0.000000,0.0,...,0.157441,0.000000,0.000000,0.006267,0.512395,0.18399,0.000000,0.142081,0.000000,TCGA-BA-4075-01
2,0.000000,0.014393,0.099051,0.133970,0.187154,0.0,0.261970,0.086587,0.277738,0.0,...,0.000000,0.141262,0.044657,0.000000,0.145821,0.00000,0.264359,0.000000,0.151672,TCGA-BA-4076-01
3,0.057826,0.000000,0.141089,0.000000,0.000000,0.0,0.172830,0.064186,0.000000,0.0,...,0.041349,0.145213,0.000000,0.000000,0.061670,0.00000,0.000000,0.000000,0.032390,TCGA-BA-4077-01
4,0.081707,0.000000,0.002183,0.304757,0.000000,0.0,0.002249,0.048936,0.000000,0.0,...,0.047431,0.000000,0.106920,0.000000,0.167654,0.00000,0.104511,0.000000,0.000000,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,0.016872,0.000000,0.028802,0.000000,0.007618,0.0,0.000000,0.133422,0.000000,0.0,...,0.003168,0.000000,0.000000,0.000000,0.095593,0.00000,0.007489,0.026297,0.141043,TCGA-UF-A7JT-01
517,0.000000,0.000000,0.096453,0.000000,0.000000,0.0,0.099151,0.114795,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.076434,0.00000,0.012930,0.045506,0.041790,TCGA-UF-A7JV-01
518,0.189707,0.099920,0.081071,0.241151,0.000000,0.0,0.076295,0.060814,0.000000,0.0,...,0.134379,0.000000,0.000000,0.000000,0.302072,0.00000,0.118916,0.000000,0.128158,TCGA-UP-A6WW-01
519,0.089585,0.000000,0.109603,0.063868,0.128936,0.0,0.184369,0.000000,0.000000,0.0,...,0.000000,0.000000,0.002656,0.000000,0.154386,0.00000,0.063998,0.000000,0.104354,TCGA-WA-A7GZ-01


In [27]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z4, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z4.shape[1])])
selected_features = coef.abs().nlargest(180).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Ze4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne4 = pd.DataFrame(Ze4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    ne4["Hugo"]=hugo
except Exception:
    pass


G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 42.406864166259766, tolerance: 37.769447326660156
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 80.39726257324219, tolerance: 37.769447326660156
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 82.83099365234375, tolerance: 37.769447326660156
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to incr

Number of selected features:  180
Selected features:  [268, 94, 156, 273, 163, 31, 295, 171, 119, 22, 245, 174, 186, 208, 219, 104, 148, 73, 246, 290, 179, 288, 123, 256, 285, 131, 270, 155, 115, 98, 220, 92, 192, 117, 132, 141, 1, 217, 272, 297, 251, 296, 211, 231, 8, 37, 242, 182, 10, 9, 124, 29, 165, 252, 257, 23, 275, 2, 158, 281, 259, 16, 149, 230, 162, 258, 129, 213, 6, 178, 173, 12, 100, 254, 184, 238, 145, 136, 223, 62, 154, 84, 196, 282, 241, 45, 140, 200, 146, 139, 96, 240, 190, 234, 11, 44, 299, 167, 248, 5, 4, 237, 264, 225, 61, 236, 71, 118, 74, 197, 109, 269, 80, 106, 218, 72, 150, 232, 51, 77, 226, 181, 168, 122, 56, 126, 66, 135, 202, 199, 283, 3, 39, 121, 249, 42, 166, 159, 18, 26, 276, 47, 102, 144, 40, 7, 13, 14, 15, 17, 19, 20, 21, 24, 25, 27, 28, 30, 32, 33, 34, 35, 36, 38, 41, 43, 46, 48, 49, 50, 52, 53, 54, 55, 57, 58, 59, 60, 63, 64]


In [28]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=180)
rfe.fit(Z4, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zre4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre4 = pd.DataFrame(Zre4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    nre4["Hugo"]=hugo
except Exception:
    pass

Selected Features: [1, 2, 3, 4, 5, 6, 7, 14, 16, 18, 20, 24, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 49, 50, 54, 55, 56, 57, 59, 60, 61, 62, 63, 66, 69, 72, 73, 75, 76, 77, 79, 80, 85, 86, 87, 89, 91, 93, 97, 99, 100, 101, 102, 103, 104, 106, 107, 108, 109, 110, 111, 116, 117, 122, 123, 125, 126, 127, 128, 129, 130, 131, 132, 133, 136, 137, 139, 140, 141, 147, 149, 155, 156, 157, 158, 159, 160, 162, 167, 169, 170, 171, 172, 175, 176, 179, 180, 181, 182, 184, 187, 188, 189, 190, 191, 192, 193, 195, 199, 201, 203, 204, 205, 207, 208, 209, 215, 216, 217, 218, 220, 221, 222, 224, 225, 226, 227, 228, 229, 230, 231, 233, 234, 236, 237, 238, 239, 241, 242, 243, 244, 246, 247, 248, 250, 252, 254, 257, 258, 260, 261, 264, 267, 268, 272, 274, 276, 279, 284, 285, 287, 290, 293, 294, 295, 297, 298, 299, 300]


In [29]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z4, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z4.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(180)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zrf4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf4 = pd.DataFrame(Zrf4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nrf4["Hugo"]=hugo
nrf4

Number of selected features:  180
Selected features:  [51, 22, 123, 96, 3, 197, 201, 31, 179, 119, 92, 219, 299, 143, 257, 280, 139, 162, 157, 131, 129, 256, 223, 268, 94, 171, 273, 100, 288, 12, 158, 249, 37, 246, 250, 275, 69, 57, 23, 62, 200, 239, 184, 168, 218, 191, 165, 271, 296, 195, 80, 124, 269, 251, 172, 70, 178, 276, 151, 285, 59, 177, 114, 84, 245, 2, 166, 180, 90, 36, 186, 289, 74, 182, 203, 77, 261, 211, 113, 286, 287, 9, 136, 156, 194, 187, 35, 42, 126, 161, 232, 164, 295, 26, 263, 148, 32, 40, 173, 241, 290, 259, 97, 145, 217, 104, 236, 258, 281, 144, 45, 213, 109, 167, 24, 174, 192, 64, 163, 196, 117, 44, 142, 169, 208, 28, 150, 260, 283, 53, 297, 71, 50, 135, 141, 41, 43, 52, 272, 19, 300, 61, 86, 202, 214, 130, 133, 270, 18, 11, 230, 298, 265, 122, 293, 199, 244, 4, 262, 88, 155, 78, 240, 55, 91, 73, 231, 93, 25, 220, 242, 247, 185, 176, 255, 87, 29, 229, 152, 146]


,0,1,2,3,4,5,6,7,8,9,...,131,132,133,134,135,136,137,138,139,Hugo
0,0.000000,0.099348,0.146631,0.016639,0.000000,0.000000,0.000000,0.013704,0.031467,0.0,...,0.000000,0.046157,0.000000,0.113121,0.0,0.000000,0.043429,0.009755,0.0,TCGA-BA-4074-01
1,0.000000,0.000000,0.084236,0.000000,0.000000,0.032078,0.065382,0.000000,0.085430,0.0,...,0.000000,0.000000,0.356982,0.000000,0.0,0.078391,0.010281,0.292153,0.0,TCGA-BA-4075-01
2,0.000000,0.000000,0.484669,0.080939,0.000000,0.000000,0.000000,0.029265,0.000000,0.0,...,0.106484,0.000000,0.050178,0.205965,0.0,0.000000,0.000000,0.267624,0.0,TCGA-BA-4076-01
3,0.000000,0.057826,0.207239,0.016287,0.000000,0.000000,0.000000,0.030869,0.003378,0.0,...,0.000000,0.123658,0.081118,0.247992,0.0,0.000000,0.148375,0.033829,0.0,TCGA-BA-4077-01
4,0.052665,0.081707,0.304545,0.068429,0.173410,0.000000,0.000000,0.201238,0.000000,0.0,...,0.000000,0.050378,0.000000,0.171540,0.0,0.052225,0.000000,0.000000,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,0.000000,0.016872,0.093657,0.000000,0.013718,0.000000,0.000000,0.022350,0.021835,0.0,...,0.000000,0.158446,0.019691,0.120535,0.0,0.000000,0.000000,0.085143,0.0,TCGA-UF-A7JT-01
517,0.000000,0.000000,0.075076,0.012992,0.000000,0.000608,0.000000,0.076437,0.025946,0.0,...,0.000000,0.067464,0.000000,0.124552,0.0,0.070406,0.017403,0.075078,0.0,TCGA-UF-A7JV-01
518,0.000000,0.189707,0.429602,0.065781,0.000000,0.000000,0.000000,0.209120,0.000000,0.0,...,0.000000,0.000000,0.000000,0.225622,0.0,0.000000,0.159500,0.000000,0.0,TCGA-UP-A6WW-01
519,0.000000,0.089585,0.185556,0.000000,0.000000,0.000000,0.000000,0.171248,0.040201,0.0,...,0.000000,0.000000,0.000000,0.078124,0.0,0.000000,0.158017,0.000000,0.0,TCGA-WA-A7GZ-01


Merging dataset

In [30]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(n1['Hugo'][i]==n3['Hugo'][j]):
            z.append(n1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==n4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,n1,on='Hugo')
mdf=pd.merge(mdf,n3,on='Hugo')
mdf=pd.merge(mdf,n4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_val=df_val
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
Y_val = get_target(df_val)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)
E_val = get_target(df_val)
# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)






(512, 26)
(512, 881)


In [31]:
epochs = 30
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)

Epoch 1/30
1/1 [==============================] - 26s 26s/step - loss: 554376.5625 - val_loss: 98082.4844
Epoch 2/30
1/1 [==============================] - 13s 13s/step - loss: 547799.1250 - val_loss: 98082.4453
Epoch 3/30
1/1 [==============================] - 11s 11s/step - loss: 551859.5625 - val_loss: 98082.4141
Epoch 4/30
1/1 [==============================] - 15s 15s/step - loss: 548037.7500 - val_loss: 98082.3750
Epoch 5/30
1/1 [==============================] - 16s 16s/step - loss: 575037.4375 - val_loss: 98082.3359
Epoch 6/30
1/1 [==============================] - 13s 13s/step - loss: 548037.6875 - val_loss: 98082.2969
Epoch 7/30
1/1 [==============================] - 12s 12s/step - loss: 547779.6875 - val_loss: 98082.2656
Epoch 8/30
1/1 [==============================] - 10s 10s/step - loss: 547910.3125 - val_loss: 98082.2266
Epoch 9/30
1/1 [==============================] - 10s 10s/step - loss: 548037.5625 - val_loss: 98082.1875
Epoch 10/30
1/1 [=============================

In [36]:
import deepsurvk

dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)


TypeError: common_callbacks() missing 3 required positional arguments: 'x_val', 'y_val', and 'event_val'

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

In [34]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nl1['Hugo'][i]==nl3['Hugo'][j]):
            z.append(nl1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nl4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nl1,on='Hugo')
mdf=pd.merge(mdf,nl3,on='Hugo')
mdf=pd.merge(mdf,nl4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)


epochs = 10
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv)")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)



(512, 26)
(512, 529)
Epoch 1/10
1/1 [==============================] - ETA: 0s - loss: 571239.0625

ValueError: in user code:

    File "C:\Users\Pro Deepali\AppData\Roaming\Python\Python39\site-packages\keras\engine\training.py", line 1525, in test_function  *
        return step_function(self, iterator)
    File "C:\Users\Pro Deepali\AppData\Roaming\Python\Python39\site-packages\keras\engine\training.py", line 1514, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "C:\Users\Pro Deepali\AppData\Roaming\Python\Python39\site-packages\keras\engine\training.py", line 1507, in run_step  **
        outputs = model.test_step(data)
    File "C:\Users\Pro Deepali\AppData\Roaming\Python\Python39\site-packages\keras\engine\training.py", line 1471, in test_step
        y_pred = self(x, training=False)
    File "C:\Users\Pro Deepali\AppData\Roaming\Python\Python39\site-packages\keras\utils\traceback_utils.py", line 67, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "C:\Users\Pro Deepali\AppData\Roaming\Python\P

In [ ]:
result = np.zeros((5, 4, 5))

result[0][0][0]=c_index_test
result[0][0][1]=mse
result[0][0][2]=rmse
result[0][0][3]=mae
result[0][0][4]=mdae

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][0][0]=c_index
result[1][0][1]=mse
result[1][0][2]=r2
result[1][0][3]=mae
result[1][0][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][0][0]=c_index
result[2][0][1]=mse
result[2][0][2]=r2
result[2][0][3]=mae
result[2][0][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][0][0]=c_index
result[3][0][1]=mse
result[3][0][2]=r2
result[3][0][3]=mae
result[3][0][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][0][0]=c_index
result[4][0][1]=mse
result[4][0][2]=r2
result[4][0][3]=mae
result[4][0][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(ne1['Hugo'][i]==ne3['Hugo'][j]):
            z.append(ne1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==ne4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,ne1,on='Hugo')
mdf=pd.merge(mdf,ne3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][1][0]=c_index_test
result[0][1][1]=mse
result[0][1][2]=rmse
result[0][1][3]=mae
result[0][1][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][1][0]=c_index
result[1][1][1]=mse
result[1][1][2]=r2
result[1][1][3]=mae
result[1][1][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][1][0]=c_index
result[2][1][1]=mse
result[2][1][2]=r2
result[2][1][3]=mae
result[2][1][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][1][0]=c_index
result[3][1][1]=mse
result[3][1][2]=r2
result[3][1][3]=mae
result[3][1][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][1][0]=c_index
result[4][1][1]=mse
result[4][1][2]=r2
result[4][1][3]=mae
result[4][1][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nre1['Hugo'][i]==nre3['Hugo'][j]):
            z.append(nre1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nre4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nre1,on='Hugo')
mdf=pd.merge(mdf,nre3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][2][0]=c_index_test
result[0][2][1]=mse
result[0][2][2]=rmse
result[0][2][3]=mae
result[0][2][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][2][0]=c_index
result[1][2][1]=mse
result[1][2][2]=r2
result[1][2][3]=mae
result[1][2][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][2][0]=c_index
result[2][2][1]=mse
result[2][2][2]=r2
result[2][2][3]=mae
result[2][2][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][2][0]=c_index
result[3][2][1]=mse
result[3][2][2]=r2
result[3][2][3]=mae
result[3][2][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][2][0]=c_index
result[4][2][1]=mse
result[4][2][2]=r2
result[4][2][3]=mae
result[4][2][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nrf1['Hugo'][i]==nrf3['Hugo'][j]):
            z.append(nrf1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nrf4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nrf1,on='Hugo')
mdf=pd.merge(mdf,nrf3,on='Hugo')
mdf=pd.merge(mdf,nrf4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][3][0]=c_index_test
result[0][3][1]=mse
result[0][3][2]=rmse
result[0][3][3]=mae
result[0][3][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][3][0]=c_index
result[1][3][1]=mse
result[1][3][2]=r2
result[1][3][3]=mae
result[1][3][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][3][0]=c_index
result[2][3][1]=mse
result[2][3][2]=r2
result[2][3][3]=mae
result[2][3][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][3][0]=c_index
result[3][3][1]=mse
result[3][3][2]=r2
result[3][3][3]=mae
result[3][3][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][3][0]=c_index
result[4][3][1]=mse
result[4][3][2]=r2
result[4][3][3]=mae
result[4][3][4]=mdae

In [ ]:
!pip install pandas openpyxl
data_0_0 = result[0, 0:4, 0]
data_1_0 = result[3, 0:4, 0]
data_2_0 = result[2, 0:4, 0]
data_3_0 = result[1, 0:4, 0]
data_4_0 = result[4, 0:4, 0]
combined_data_0 = np.concatenate((data_0_0, data_1_0, data_2_0, data_3_0, data_4_0))
data_0_1 = result[0, 0:4, 1]
data_1_1 = result[3, 0:4, 1]q
data_2_1 = result[2, 0:4, 1]
data_3_1 = result[1, 0:4, 1]
data_4_1 = result[4, 0:4, 1]
combined_data_1 = np.concatenate((data_0_1, data_1_1, data_2_1, data_3_1, data_4_1))
data_0_2 = result[0, 0:4, 2]
data_1_2 = result[3, 0:4, 2]
data_2_2 = result[2, 0:4, 2]
data_3_2 = result[1, 0:4, 2]
data_4_2 = result[4, 0:4, 2]
combined_data_2 = np.concatenate((data_0_2, data_1_2, data_2_2, data_3_2, data_4_2))
data_0_3 = result[0, 0:4, 3]
data_1_3 = result[3, 0:4, 3]
data_2_3 = result[2, 0:4, 3]
data_3_3 = result[1, 0:4, 3]
data_4_3 = result[4, 0:4, 3]
combined_data_3 = np.concatenate((data_0_3, data_1_3, data_2_3, data_3_3, data_4_3))
data_0_4 = result[0, 0:4, 4]
data_1_4 = result[3, 0:4, 4]
data_2_4 = result[2, 0:4, 4]
data_3_4 = result[1, 0:4, 4]
data_4_4 = result[4, 0:4, 4]
combined_data_4 = np.concatenate((data_0_4, data_1_4, data_2_4, data_3_4, data_4_4))
# Create a DataFrame
df = pd.DataFrame({
    'Combined_Data_0': combined_data_0,
    'Combined_Data_1': combined_data_1,
    'Combined_Data_2': combined_data_2,
    'Combined_Data_3': combined_data_3,
    'Combined_Data_4': combined_data_4
})

# Save the DataFrame to an Excel file
df.to_excel('combined_data.xlsx', index=False)

print("Data has been saved to combined_data.xlsx")